# Lab 4: HyDE - Hypothetical Document Embedding

**Level:** Basic | **Duration:** ~35 minutes

## What You'll Learn
- The problem with embedding raw questions for retrieval
- How HyDE works: generate a hypothetical answer, then embed that instead
- When HyDE improves retrieval and when it doesn't
- How to implement HyDE with LangChain
- How to compare retrieval quality between naive RAG and HyDE

## The Core Insight
Questions and answers live in different parts of embedding space. A question like "What is the refund policy?" is semantically distant from the actual policy text. HyDE bridges this gap by first generating a hypothetical answer, which is closer in embedding space to the real document.

## Setup

In [ ]:
!pip install -q langchain langchain-google-genai langchain-chroma chromadb

In [ ]:
import os
os.environ["GOOGLE_API_KEY"] = "your-gemini-key-here"

from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.documents import Document

llm = ChatGoogleGenerativeAI(model="gemini-1.5-flash", temperature=0)
embeddings = GoogleGenerativeAIEmbeddings(model="models/text-embedding-004")

print("Setup complete!")

## Step 1: Knowledge Base

We'll use a set of airline operations documents as our knowledge base.

In [ ]:
knowledge_base = """
# SkyWing Airlines Operations Manual

## Refund and Cancellation Policy

Passengers may cancel their booking up to 24 hours before departure for a full refund if the ticket was purchased at least 7 days before the flight. Cancellations made less than 24 hours before departure are subject to a cancellation fee of 75 euros for short-haul flights and 150 euros for long-haul flights. Non-refundable tickets (Light fare) are not eligible for refunds but may be converted to travel credit valid for 12 months. Travel credit can be used on any SkyWing flight and is transferable to other passengers.

## Rebooking Policy

Passengers on Flex and Premium fares can rebook their flights free of charge up to 3 hours before departure. Light fare passengers can rebook for a fee of 50 euros plus any fare difference. In case of flight cancellation by SkyWing, all passengers are automatically rebooked on the next available flight at no additional cost. Passengers may also choose a full refund instead of rebooking if the cancellation is initiated by the airline.

## Irregular Operations (IRROPS)

When flights are delayed by more than 2 hours, SkyWing provides complimentary refreshments. Delays exceeding 4 hours include a meal voucher worth 15 euros. For overnight delays, hotel accommodation and ground transport are provided for passengers who are not local residents. The operations control center coordinates rebooking and communicates updates via SMS, email, and the SkyWing mobile app.

## Upgrade Policy

Operational upgrades are offered at the gate when economy class is oversold. Priority is given to frequent flyer members by tier status (Platinum > Gold > Silver). Paid upgrades are available through the mobile app starting 72 hours before departure. The upgrade price is calculated as the fare difference minus 20% loyalty discount for frequent flyer members.

## Special Assistance

Passengers with reduced mobility should request assistance at least 48 hours before departure. SkyWing provides wheelchair service, priority boarding, and adapted seating at no extra charge. Service animals are permitted in the cabin on all flights with prior notification. Passengers who are deaf or hard of hearing receive all announcements via text on the in-flight entertainment screen.

## Crew Scheduling

Flight crew must have a minimum rest period of 12 hours between duties. Maximum duty time is 13 hours for two-pilot crews and 17 hours for augmented crews (3+ pilots). Crew scheduling is managed through the Sky Suite scheduling system. Reserve crew are assigned monthly bid lines and must be available within 90 minutes of callout when on standby.

## Fuel Policy

All flights carry fuel calculated as: trip fuel + 5% contingency + alternate airport fuel + 30 minutes holding fuel + captain's discretionary fuel. The fuel management system optimizes tankering decisions based on fuel price differentials between airports. Captains have final authority on fuel load and may request additional fuel based on weather conditions or anticipated delays.
"""

# Split and index
splitter = RecursiveCharacterTextSplitter(chunk_size=400, chunk_overlap=50)
chunks = splitter.split_text(knowledge_base)
docs = [Document(page_content=c, metadata={"chunk_id": i}) for i, c in enumerate(chunks)]

vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    collection_name="hyde_test",
)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print(f"Indexed {len(chunks)} chunks.")

## Step 2: Baseline Naive RAG

First, let's build a standard RAG pipeline that embeds the question directly.

In [ ]:
answer_prompt = ChatPromptTemplate.from_template("""
Answer the question based ONLY on the following context.
If the context doesn't contain the answer, say so.

Context:
{context}

Question: {question}

Answer:
""")

def format_docs(docs):
    return "\n\n---\n\n".join(doc.page_content for doc in docs)

def naive_rag(question):
    """Standard RAG: embed the question directly, retrieve, generate."""
    retrieved = retriever.invoke(question)
    context = format_docs(retrieved)
    answer = (answer_prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )
    return {
        "answer": answer,
        "retrieved_chunks": [doc.page_content[:100] for doc in retrieved],
    }

# Test
result = naive_rag("What is the refund policy?")
print(f"Answer: {result['answer']}\n")
print("Retrieved chunks:")
for i, chunk in enumerate(result["retrieved_chunks"], 1):
    print(f"  {i}. {chunk}...")

## Step 3: Implement HyDE

HyDE works in three steps:
1. **Generate** a hypothetical answer to the question (using the LLM without retrieval)
2. **Embed** the hypothetical answer instead of the original question
3. **Retrieve** using the hypothetical answer's embedding

The key insight: a hypothetical answer about refund policies will be semantically closer to the actual refund policy text than the question "What is the refund policy?" is.

In [ ]:
hyde_prompt = ChatPromptTemplate.from_template("""
Write a short, factual paragraph that would answer the following question.
Write it as if it were part of an official document or manual.
Do NOT say "I don't know" — generate a plausible answer even if you're unsure.

Question: {question}

Hypothetical answer:
""")

def hyde_rag(question):
    """HyDE RAG: generate hypothetical answer, embed that, retrieve, generate final answer."""
    # Step 1: Generate hypothetical answer
    hypothetical = (hyde_prompt | llm | StrOutputParser()).invoke(
        {"question": question}
    )

    # Step 2: Embed the hypothetical answer and retrieve
    hyde_embedding = embeddings.embed_query(hypothetical)
    retrieved = vectorstore.similarity_search_by_vector(hyde_embedding, k=3)

    # Step 3: Generate final answer from retrieved context
    context = format_docs(retrieved)
    answer = (answer_prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )

    return {
        "hypothetical": hypothetical,
        "answer": answer,
        "retrieved_chunks": [doc.page_content[:100] for doc in retrieved],
    }

# Test
result = hyde_rag("What is the refund policy?")
print(f"Hypothetical answer: {result['hypothetical'][:200]}...\n")
print(f"Final answer: {result['answer']}\n")
print("Retrieved chunks:")
for i, chunk in enumerate(result["retrieved_chunks"], 1):
    print(f"  {i}. {chunk}...")

## Step 4: Head-to-Head Comparison

Let's run both pipelines on 5 questions and compare the retrieval results side by side.

In [ ]:
test_questions = [
    "Can I get my money back if I cancel a cheap ticket?",
    "What happens when a flight is cancelled due to weather?",
    "How do pilots decide how much fuel to load?",
    "I need a wheelchair at the airport, what should I do?",
    "How can I get upgraded to business class?",
]

for q in test_questions:
    print(f"\n{'='*70}")
    print(f"QUESTION: {q}")
    print(f"{'='*70}")

    naive_result = naive_rag(q)
    hyde_result = hyde_rag(q)

    print(f"\n--- NAIVE RAG ---")
    print(f"Answer: {naive_result['answer'][:200]}")
    print(f"Chunks: {[c[:60]+'...' for c in naive_result['retrieved_chunks']]}")

    print(f"\n--- HyDE RAG ---")
    print(f"Hypothetical: {hyde_result['hypothetical'][:150]}...")
    print(f"Answer: {hyde_result['answer'][:200]}")
    print(f"Chunks: {[c[:60]+'...' for c in hyde_result['retrieved_chunks']]}")

## Step 5: Measure Retrieval Overlap

Let's quantify how different the retrieval results are between the two approaches.

In [ ]:
def get_chunk_ids(result):
    """Extract chunk content fingerprints for comparison."""
    return set(result["retrieved_chunks"])

print(f"{'Question':<55} {'Overlap':<10} {'Naive unique':<15} {'HyDE unique':<15}")
print("-" * 95)

for q in test_questions:
    naive_result = naive_rag(q)
    hyde_result = hyde_rag(q)

    naive_chunks = get_chunk_ids(naive_result)
    hyde_chunks = get_chunk_ids(hyde_result)

    overlap = len(naive_chunks & hyde_chunks)
    naive_only = len(naive_chunks - hyde_chunks)
    hyde_only = len(hyde_chunks - naive_chunks)

    print(f"{q[:53]:<55} {overlap:<10} {naive_only:<15} {hyde_only:<15}")

print("\nWhen HyDE retrieves different chunks, check if they're more relevant.")

## Step 6: When HyDE Helps (and When It Doesn't)

HyDE is not universally better. Let's test cases where it might hurt.

In [ ]:
# Cases where HyDE might NOT help:
tricky_questions = [
    # Factoid with specific number - LLM might hallucinate wrong number
    "What is the exact cancellation fee for a long-haul flight?",
    # Very specific terminology - original query might match better
    "What is the IRROPS procedure?",
    # Question about absence of information
    "Does the policy mention anything about cryptocurrency refunds?",
]

for q in tricky_questions:
    print(f"\nQ: {q}")
    print("-" * 60)

    hyde_result = hyde_rag(q)
    naive_result = naive_rag(q)

    print(f"Hypothetical (may contain errors): {hyde_result['hypothetical'][:150]}...")
    print(f"\nNaive answer: {naive_result['answer'][:150]}")
    print(f"HyDE answer:  {hyde_result['answer'][:150]}")

print("\n" + "="*60)
print("KEY INSIGHT: HyDE works best when:")
print("  + The question uses different vocabulary than the source")
print("  + The question is conceptual / paraphrased")
print("  + The knowledge base uses formal language")
print("\nHyDE may hurt when:")
print("  - The hypothetical answer contains wrong facts (misleads retrieval)")
print("  - The question uses exact terminology from the source")
print("  - The question is about absence of information")

## Step 7: Latency Comparison

HyDE adds an extra LLM call. Let's measure the overhead.

In [ ]:
import time

question = "What happens if my flight is delayed?"

# Measure naive RAG
start = time.time()
for _ in range(3):
    naive_rag(question)
naive_time = (time.time() - start) / 3

# Measure HyDE RAG
start = time.time()
for _ in range(3):
    hyde_rag(question)
hyde_time = (time.time() - start) / 3

print(f"Average latency (3 runs):")
print(f"  Naive RAG: {naive_time:.2f}s")
print(f"  HyDE RAG:  {hyde_time:.2f}s")
print(f"  Overhead:  {hyde_time - naive_time:.2f}s ({(hyde_time/naive_time - 1)*100:.0f}% slower)")
print(f"\nThe extra LLM call for hypothesis generation adds ~1-2 seconds.")

---

## YOUR TURN: Multi-Query Approach

Instead of HyDE, try a **multi-query** approach:
1. Generate 3 different phrasings of the original question
2. Retrieve top-3 chunks for each phrasing
3. Merge and deduplicate the results
4. Use the merged context for generation

Compare this with both naive RAG and HyDE.

In [ ]:
# YOUR TURN: Implement multi-query retrieval

multi_query_prompt = ChatPromptTemplate.from_template("""
Generate 3 different phrasings of the following question.
Each phrasing should approach the topic from a different angle.
Return only the 3 questions, one per line.

Original question: {question}

Alternative phrasings:
""")

def multi_query_rag(question):
    # Step 1: Generate alternative phrasings
    alternatives = (multi_query_prompt | llm | StrOutputParser()).invoke(
        {"question": question}
    )
    queries = [question] + [q.strip() for q in alternatives.strip().split("\n") if q.strip()]

    print(f"Queries used: {queries}\n")

    # Step 2: Retrieve for each query
    all_retrieved = []
    seen = set()
    for q in queries:
        results = retriever.invoke(q)
        for doc in results:
            if doc.page_content not in seen:
                seen.add(doc.page_content)
                all_retrieved.append(doc)

    # Step 3: Generate answer from merged context
    context = format_docs(all_retrieved[:5])  # Top 5 unique chunks
    answer = (answer_prompt | llm | StrOutputParser()).invoke(
        {"context": context, "question": question}
    )

    return {
        "answer": answer,
        "num_unique_chunks": len(all_retrieved),
        "queries_used": len(queries),
    }

# Test it
result = multi_query_rag("Can I get my money back if I cancel a cheap ticket?")
print(f"Answer: {result['answer']}")
print(f"\nUsed {result['queries_used']} queries, found {result['num_unique_chunks']} unique chunks.")

## Key Takeaways

1. **HyDE bridges the query-document gap** by generating a hypothetical answer first.
2. **It works best** when users phrase questions differently from how documents are written.
3. **It can hurt** when the LLM generates wrong facts in the hypothesis (misleading retrieval).
4. **The latency trade-off** is ~1-2 extra seconds per query (one additional LLM call).
5. **Multi-query** is an alternative that diversifies retrieval without risk of hallucinated hypotheses.
6. In practice, **test all three** on your data and pick what works best for your domain.

**Next:** In Lab 5, we'll improve retrieval quality with re-ranking.